## Extendable hooks through registries

Make code accessible by inserting extendable hooks using registries. A registry can be added to functions that could otherwise not be accessible to end users. End users can then extend the capabilities of the hidden functions by registering new functions.

In [ ]:
from registry_factory.registry import Registry

@Registry.register("option_1")
def option_1() -> int:
    return 1

@Registry.register("option_3")
def option_3() -> int:
    return 3

In [ ]:
def _some_hidden_function(a: str) -> int:
    try:
        return print(Registry.get(f"option_{a}")())
    except Exception as e:
        raise RuntimeError("Error getting the option", e)

In [ ]:
_some_hidden_function(1) # Returns 1
_some_hidden_function(3) # Returns 3
_some_hidden_function(2) # Error

In [ ]:
@Registry.register("option_2") # External user adds new option
def option_2() -> int:
    return 2

In [ ]:
_some_hidden_function(2)  # Returns 2

## Wrapper compatibility

Make independent versions compatible by registering them through a wrapper. Here, we provide a toy example where to functions with similar functionality but incompatible outputs are used in the same final function. This example can be retrofitted with registries and a wrapper function to work without a problem.

In [ ]:
from registry_factory.factory import Factory

class Registries(Factory):
    ModelRegistry = Factory.create_registry(name="model_registry", checks=[])

def func1():
    return "hello world"

def func2():
    return ["hello universe"]


def final_function(key: str) -> str:
    return Registries.ModelRegistry.get(key)()


In [ ]:
def wrapper_function(func):
    def wrapper(*args, **kwargs):
        out = func(*args, **kwargs)
        if type(out) is list:
            return out[0]
        else:
            return out
    return wrapper


Registries.ModelRegistry.register_prebuilt(wrapper_function(func1), "world")
Registries.ModelRegistry.register_prebuilt(wrapper_function(func2), "universe")

In [ ]:
print(final_function("world"))
print(final_function("universe"))